# Further Experiment 4 — Directionality ablation

Both the BiLSTM and DistilBERT are **bidirectional**. How much does that actually buy us? We train a **one-directional** LSTM that is identical to the BiLSTM in every other way (same data, vocab, sizes, epochs, readout) and only flip `bidirectional=False`.

**What we noticed:** bidirectionality buys essentially **nothing** here — the uni- and bi-directional models tie within seed noise — while the unidirectional model trains ~2-3x faster. On short tweets a single forward pass already captures the salient sentiment words. (So DistilBERT's edge is *pretraining*, not merely reading both directions.)

## Setup

In [ ]:
!pip install -q datasets scikit-learn torch transformers accelerate matplotlib

In [ ]:
import os, re, time, random, math
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset, concatenate_datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
def clean_tweet(text):
    text = str(text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)   # strip URLs
    text = re.sub(r'@\w+', '@user', text)                  # @mentions -> @user
    text = re.sub(r'\s+', ' ', text).strip()               # normalize whitespace
    return text

## Same 50k balanced subsample, 40k/10k split (matches the main notebooks)

In [ ]:
ds = load_dataset('contemmcm/sentiment140')['complete']
neg = ds.filter(lambda x: x['label'] == 0).shuffle(seed=SEED).select(range(25_000))
pos = ds.filter(lambda x: x['label'] == 1).shuffle(seed=SEED).select(range(25_000))
data = concatenate_datasets([neg, pos]).shuffle(seed=SEED)
X = [clean_tweet(t) for t in data['text']]; y = list(data['label'])
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(len(X_tr), len(X_te))

## BiLSTM / UniLSTM (one class, `bidir` flag) + train/eval helpers

In [ ]:
def tokenize(text):
    return re.findall(r'@\w+|#\w+|[a-zA-Z]+|[0-9]+|[^\s]', text.lower())

def build_vocab(texts, max_size=50_000, min_freq=2):
    c = Counter()
    for t in texts: c.update(tokenize(t))
    itos = ['<pad>', '<unk>'] + [w for w, f in c.most_common(max_size) if f >= min_freq]
    return {w: i for i, w in enumerate(itos)}

MAX_LEN = 50
def encode(text, stoi):
    ids = [stoi.get(tok, 1) for tok in tokenize(text)][:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))

class TweetDS(Dataset):
    def __init__(self, texts, labels, stoi):
        self.x = [encode(t, stoi) for t in texts]; self.y = list(labels)
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return torch.tensor(self.x[i]), torch.tensor(self.y[i])

class LSTMClassifier(nn.Module):
    # bidirectional flag lets us reuse this for the directionality ablation
    def __init__(self, vocab_size, emb=100, hid=128, bidir=True):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=0)
        self.lstm = nn.LSTM(emb, hid, batch_first=True, bidirectional=bidir)
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(hid * (2 if bidir else 1), 2)
        self.bidir = bidir
    def forward(self, x):
        _, (h, _) = self.lstm(self.emb(x))
        h = torch.cat([h[-2], h[-1]], dim=1) if self.bidir else h[-1]
        return self.fc(self.drop(h))

def train_lstm(texts, labels, stoi, epochs=3, bidir=True):
    model = LSTMClassifier(len(stoi), bidir=bidir).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    dl = DataLoader(TweetDS(texts, labels, stoi), batch_size=128, shuffle=True)
    model.train()
    for _ in range(epochs):
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad(); loss = loss_fn(model(xb), yb); loss.backward(); opt.step()
    return model

@torch.no_grad()
def eval_lstm(model, texts, labels, stoi):
    model.eval(); preds = []
    dl = DataLoader(TweetDS(texts, labels, stoi), batch_size=512)
    for xb, _ in dl:
        preds.append(model(xb.to(device)).argmax(1).cpu().numpy())
    preds = np.concatenate(preds)
    return accuracy_score(labels, preds), f1_score(labels, preds)

## Train both, identically, and compare

In [ ]:
stoi = build_vocab(X_tr)   # same vocab for both
rows = []
for bidir, name in [(True, 'BiLSTM (bidirectional)'), (False, 'UniLSTM (one-directional)')]:
    torch.manual_seed(SEED)
    t0 = time.time(); model = train_lstm(X_tr, y_tr, stoi, epochs=3, bidir=bidir)
    train_time = time.time() - t0
    acc, f1 = eval_lstm(model, X_te, y_te, stoi)
    rows.append({'model': name, 'accuracy': round(acc, 4), 'f1': round(f1, 4),
                 'train_time_s': round(train_time, 1)})
    print(rows[-1])
pd.DataFrame(rows)

## What we found

| LSTM | Accuracy | F1 | Train time |
|---|---|---|---|
| BiLSTM (bidirectional) | 0.7644 | 0.7704 | 85 s |
| UniLSTM (one-directional) | 0.7688 | 0.7664 | 31 s |

The ~0.4-point differences are within seed noise and even split between the two (one wins on accuracy, the other on F1), so **bidirectionality made no meaningful difference** — at ~2.7x the training cost. The backward pass adds little on short, noisy tweets.